# 🎮 Video Games Sentiment Analysis (Analisis Sentimen Ulasan Video Games)

Notebook ini mengimplementasikan alur lengkap analisis sentimen berbasis **Natural Language Processing (NLP)** dan **Machine Learning** pada ulasan video game. Proyek ini mencakup:
1. **Persiapan Lingkungan & Pemuatan Pustaka**
2. **Pemuatan Dataset** (Termasuk utilitas streaming file besar via `requests` & `gzip`)
3. **Exploratory Data Analysis (EDA)**
4. **Pra-pemrosesan Teks (Text Preprocessing)**
5. **Ekstraksi Fitur (TF-IDF Vectorization)**
6. **Pelatihan Model Klasifikasi (Logistic Regression Pipeline)**
7. **Evaluasi Model** (Accuracy, Precision, Recall, F1-Score & Confusion Matrix)
8. **Penyimpanan Model** (`model/model_sentimen.joblib`)
9. **Pengujian Inferensi Real-time**

In [1]:
# 1. Pustaka & Dependensi
import os
import re
import gzip
import json
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
print("Semua pustaka berhasil dimuat!")

Semua pustaka berhasil dimuat!


## 2. Pemuatan Data (Data Loading)
Di bawah ini disediakan fungsi untuk membaca file besar terkompresi (`.json.gz` ulasan video games dari Amazon atau repositori publik) secara streaming tanpa membebani memori RAM, serta memuat dataset ulasan lokal.

In [2]:
def load_compressed_json_stream(file_path_or_url, max_records=5000):
    """
    Membaca berkas .json.gz besar secara streaming baris-per-baris.
    Cocok untuk dataset ulasan skala besar (seperti Amazon Video Games Review Dataset).
    """
    records = []
    if file_path_or_url.startswith("http://") or file_path_or_url.startswith("https://"):
        resp = requests.get(file_path_or_url, stream=True)
        with gzip.GzipFile(fileobj=resp.raw) as gz:
            for i, line in enumerate(gz):
                if i >= max_records:
                    break
                data = json.loads(line.decode("utf-8"))
                records.append(data)
    elif os.path.exists(file_path_or_url):
        with gzip.open(file_path_or_url, "rt", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= max_records:
                    break
                records.append(json.loads(line))
    return pd.DataFrame(records)

# Memuat dataset ulasan video games lokal
data_path = os.path.join("data", "video_games_reviews.csv")
df = pd.read_csv(data_path)
print(f"Total data ulasan: {len(df)} baris")
df.head()

Total data ulasan: 210 baris


## 3. Exploratory Data Analysis (EDA)
Memeriksa distribusi kelas sentimen (Positif = 1, Negatif = 0) dan statistik panjang teks ulasan.

In [3]:
# Cek distribusi sentimen
sentiment_counts = df["sentiment"].value_counts()
print("Distribusi Sentimen:")
print(sentiment_counts)

# Hitung panjang karakter dan kata
df["char_length"] = df["review_text"].apply(len)
df["word_count"] = df["review_text"].apply(lambda x: len(x.split()))
df[["char_length", "word_count"]].describe()

In [4]:
# Visualisasi Distribusi Sentimen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot distribusi sentimen
labels = ["Negatif (0)", "Positif (1)"]
sns.barplot(x=labels, y=sentiment_counts.values, ax=axes[0], palette=["#e74c3c", "#2ecc71"])
axes[0].set_title("Distribusi Label Sentimen Ulasan Game", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Jumlah Ulasan")

# Histogram panjang kata
sns.histplot(data=df, x="word_count", hue="sentiment", kde=True, ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_title("Distribusi Jumlah Kata per Ulasan", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Jumlah Kata")

plt.tight_layout()
plt.show()

## 4. Pra-pemrosesan Teks (Preprocessing)
Membersihkan karakter khusus, tanda baca berlebih, dan konversi ke huruf kecil (lowercase).

In [5]:
def clean_text(text):
    """Membersihkan teks ulasan dari simbol, URL, dan karakter khusus."""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review_text"].apply(clean_text)
df[["review_text", "clean_review", "sentiment"]].head()

## 5. Pembagian Data (Train & Test Split)
Membagi data ulasan menjadi 80% data latih (training) dan 20% data uji (testing) secara terstratifikasi.

In [6]:
X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Latih (Train): {len(X_train)} sampel")
print(f"Data Uji (Test)   : {len(X_test)} sampel")

Data Latih (Train): 168 sampel
Data Uji (Test)   : 42 sampel


## 6. Pipeline Pemodelan (TF-IDF + Logistic Regression)
Membangun pipeline scikit-learn terpadu yang menggabungkan ekstraksi fitur `TfidfVectorizer` dan algoritma klasifikasi `LogisticRegression`.

In [7]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

# Pelatihan model
pipeline.fit(X_train, y_train)
print("Model Pipeline berhasil dilatih!")

Model Pipeline berhasil dilatih!


## 7. Evaluasi Model
Menghitung metrik performa: Akurasi, Precision, Recall, F1-Score, dan visualisasi matriks kebingungan (Confusion Matrix).

In [8]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Akurasi Model: {acc * 100:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=["Negatif (0)", "Positif (1)"]))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Negatif", "Pred Positif"],
            yticklabels=["Aktual Negatif", "Aktual Positif"])
plt.title("Confusion Matrix Evaluasi Model Sentimen", fontsize=12, fontweight="bold")
plt.ylabel("Kelas Aktual")
plt.xlabel("Kelas Prediksi")
plt.tight_layout()
plt.show()

## 8. Penyimpanan Model (Model Serialization)
Menyimpan pipeline lengkap yang sudah terlatih ke dalam folder `model/model_sentimen.joblib` agar siap digunakan untuk deployment atau inferensi mandiri.

In [9]:
os.makedirs("model", exist_ok=True)
model_output_path = os.path.join("model", "model_sentimen.joblib")
joblib.dump(pipeline, model_output_path)
print(f"Model berhasil disimpan di: {model_output_path}")

Model berhasil disimpan di: model\model_sentimen.joblib


## 9. Pengujian Inferensi Real-time (Uji Model dengan Ulasan Baru)
Memuat model tersimpan dan menguji ulasan baru dari pemain game untuk memvalidasi performa klasifikasi.

In [10]:
# Muat model kembali dari berkas .joblib
loaded_model = joblib.load(model_output_path)

contoh_ulasan = [
    "The storyline and visual effects are mind blowing! Combat feels responsive and rewarding.",
    "Constant stuttering and crashes after the latest patch. Unplayable mess!",
    "Great multiplayer co-op experience, had hours of fun with my squad.",
    "Total cash grab with greedy microtransactions everywhere and pay to win mechanics."
]

prediksi = loaded_model.predict(contoh_ulasan)
probabilitas = loaded_model.predict_proba(contoh_ulasan)

for teks, pred, prob in zip(contoh_ulasan, prediksi, probabilitas):
    label = "POSITIF" if pred == 1 else "NEGATIF"
    confidence = max(prob) * 100
    print(f"Ulasan    : \"{teks}\"")
    print(f"Sentimen  : {label} (Keyakinan: {confidence:.2f}%)")
    print("-" * 70)